[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C51_Data_Augmentation_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与「三重检验」热身

本课全程 **纯 numpy + 标准库、CPU、不联网**。用**可控参数的模拟增强器**替代真实翻译/LLM——
因为本课要教的判断力只依赖「增强前后的样本对」，不依赖增强器的具体实现。
而且参数可控让你能做真实环境里做不到的实验（比如「把教师偏差从 0 调到 0.3」）。

这个 notebook 做三件事：① 环境自检；② 建立本课的**规则可判定任务**（这样标签保真度能精确计算）；
③ 实现贯穿全课的**三重检验**：保真度 / 多样性 / 有效性。

## 1 · 环境自检

In [ ]:
import sys, platform, math, random, itertools, collections
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np; print('numpy', np.__version__)
try:
    import pandas as pd; print('pandas', pd.__version__, '(可选)')
except Exception:
    print('pandas 未安装（可选）')
print('环境就绪 ✅  —— 本课不需要 GPU / 翻译模型 / LLM API / 联网')

## 2 · 一个规则可判定的任务：让「标签保真度」可精确计算

真实任务里，「增强后标签还对不对」需要人工判断。本课用一个**规则可判定**的合成任务绕开这个问题：

> **标签 = 情感极性**，由一条明确规则决定：
> 句中含**正面词**则为正面；若同时含**否定词**且否定词在正面词之前 3 个词内，则翻转为负面。

这样任何增强后的句子，我们都能**精确算出它的真标签**，从而量化「增强破坏了多少标签」。

In [ ]:
POS_WORDS = {'好吃', '不错', '推荐', '干净', '很好', '满意', '喜欢'}
NEG_WORDS = {'难吃', '差', '脏', '失望', '糟糕'}
NEGATORS  = {'不', '没', '别', '不太', '并不'}
NEUTRAL   = ['这家', '店', '的', '菜', '服务', '环境', '价格', '味道', '朋友', '下次',
             '我们', '昨天', '一起', '去', '吃', '了', '感觉', '整体', '还', '挺']

def rule_label(tokens, window=3):
    '''规则标签：1=正面, 0=负面。否定词在正面词前 window 个词内则翻转。'''
    score = 0
    for i, t in enumerate(tokens):
        if t in POS_WORDS:
            negated = any(tokens[j] in NEGATORS for j in range(max(0, i - window), i))
            score += -1 if negated else 1
        elif t in NEG_WORDS:
            negated = any(tokens[j] in NEGATORS for j in range(max(0, i - window), i))
            score += 1 if negated else -1
    return 1 if score > 0 else 0

def make_sentence(rng, label=None, length=10):
    '''生成一个句子及其规则标签。'''
    for _ in range(200):
        toks = list(rng.choice(NEUTRAL, size=length - 2, replace=True))
        pos = int(rng.integers(1, len(toks)))
        if rng.random() < 0.5:
            toks.insert(pos, str(rng.choice(sorted(POS_WORDS))))
        else:
            toks.insert(pos, str(rng.choice(sorted(NEG_WORDS))))
        if rng.random() < 0.4:
            toks.insert(max(0, pos - int(rng.integers(1, 3))), str(rng.choice(sorted(NEGATORS))))
        y = rule_label(toks)
        if label is None or y == label:
            return toks, y
    return toks, rule_label(toks)

rng = np.random.default_rng(0)
for _ in range(4):
    toks, y = make_sentence(rng)
    print(f'{"正面" if y else "负面"} | {" ".join(toks)}')

# 规则的关键性质：否定词的位置决定标签
assert rule_label(['这家', '店', '好吃']) == 1
assert rule_label(['这家', '店', '不', '好吃']) == 0, '否定词在窗口内 -> 翻转'
assert rule_label(['不', '这家', '店', '的', '菜', '好吃']) == 1, '否定词太远 -> 不翻转'
assert rule_label(['难吃']) == 0 and rule_label(['不', '难吃']) == 1
print('\n✅ 规则可判定：任何增强后的句子，我们都能算出它的**真**标签。')
print('   这让「增强破坏了多少标签」从主观判断变成一个可精确计算的数字。')

## 3 · 检验一：保真度（fidelity）—— 增强有没有破坏标签

In [ ]:
def fidelity(pairs):
    '''pairs: [(原tokens, 原标签, 增强后tokens)]。返回增强后规则标签仍等于原标签的比例。'''
    if not pairs: return 1.0
    keep = sum(1 for orig, y, aug in pairs if rule_label(aug) == y)
    return keep / len(pairs)

def broken_examples(pairs, k=3):
    return [(orig, y, aug) for orig, y, aug in pairs if rule_label(aug) != y][:k]

# 一个刻意危险的增强：随机删除一个词（可能删掉否定词！）
def random_deletion(tokens, p, rng):
    kept = [t for t in tokens if rng.random() >= p]
    return kept if kept else tokens[:1]

data = [make_sentence(np.random.default_rng(s)) for s in range(400)]
print(f"{'删除概率 p':>11s} {'标签保真度':>11s} {'破坏率':>9s}")
for p in [0.0, 0.05, 0.1, 0.2, 0.4]:
    r = np.random.default_rng(7)
    pairs = [(t, y, random_deletion(t, p, r)) for t, y in data]
    f = fidelity(pairs)
    print(f'{p:>11.2f} {f:>11.1%} {1-f:>9.1%}')

r = np.random.default_rng(7)
pairs20 = [(t, y, random_deletion(t, 0.2, r)) for t, y in data]
f0 = fidelity([(t, y, t) for t, y in data])
assert f0 == 1.0, '不增强时保真度必须是 1'
assert fidelity(pairs20) < 0.98, '删除 20% 的词会破坏一部分标签'
print(f'\n被破坏的例子（删除 p=0.2）:')
for orig, y, aug in broken_examples(pairs20):
    print(f'  原({"正" if y else "负"}): {" ".join(orig)}')
    print(f'  增({"正" if rule_label(aug) else "负"}): {" ".join(aug)}   ← 标签翻转了')
print('\n✅ 这就是文本增强与图像增强的根本区别：**旋转一张猫的照片它还是猫，')
print('   但删掉一个「不」字，标签就反了。**')

## 4 · 检验二：多样性（diversity）—— 新样本有多新

In [ ]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def distinct_n(texts, n=2):
    '''distinct-n：不同 n-gram 数 / 总 n-gram 数。越高越多样。'''
    total, uniq = 0, set()
    for t in texts:
        g = ngrams(t, n); total += len(g); uniq.update(g)
    return len(uniq) / total if total else 0.0

def self_bleu(texts, n=2, sample=60, seed=0):
    '''self-BLEU（简化版）：每条与其他条的 n-gram 重叠precision 的均值。越**低**越多样。'''
    r = np.random.default_rng(seed)
    idx = r.choice(len(texts), size=min(sample, len(texts)), replace=False)
    scores = []
    for i in idx:
        gi = collections.Counter(ngrams(texts[i], n))
        if not gi: continue
        others = collections.Counter()
        for j in idx:
            if j != i: others.update(ngrams(texts[j], n))
        overlap = sum(min(c, others[g]) for g, c in gi.items())
        scores.append(overlap / sum(gi.values()))
    return float(np.mean(scores)) if scores else 0.0

def cheap_embed(tokens, dim=32, seed=0):
    '''廉价嵌入：词袋 + 固定随机投影。度量的**性质**与真实嵌入一致。'''
    r = np.random.default_rng(seed)
    vocab = sorted(set(NEUTRAL) | POS_WORDS | NEG_WORDS | NEGATORS)
    proj = r.normal(size=(len(vocab), dim))
    v2i = {w: i for i, w in enumerate(vocab)}
    vec = np.zeros(dim)
    for t in tokens:
        if t in v2i: vec += proj[v2i[t]]
    n = np.linalg.norm(vec)
    return vec / n if n > 0 else vec

def embedding_coverage(texts, n_bins=8, seed=0):
    '''嵌入覆盖：把嵌入投到 2 维后统计占据了多少网格。越高覆盖越广。'''
    E = np.stack([cheap_embed(t, seed=seed) for t in texts])
    xy = E[:, :2]
    lo, hi = xy.min(0), xy.max(0)
    span = np.where(hi - lo > 1e-9, hi - lo, 1.0)
    cells = set(map(tuple, np.floor((xy - lo) / span * (n_bins - 1e-9)).astype(int)))
    return len(cells) / (n_bins * n_bins)

texts = [t for t, _ in data]
# 三组对照：原始 / 原始的近似副本（低多样性）/ 完全随机（高多样性）
dup = [t[:] for t in texts[:100] for _ in range(4)][:400]
rand_texts = [list(np.random.default_rng(1000+i).choice(NEUTRAL + sorted(POS_WORDS), size=10))
              for i in range(400)]
print(f"{'语料':<16s} {'distinct-2':>11s} {'self-BLEU-2':>12s} {'嵌入覆盖':>9s}")
for name, ts in [('原始', texts), ('近似副本', dup), ('完全随机', rand_texts)]:
    print(f'{name:<16s} {distinct_n(ts,2):>11.4f} {self_bleu(ts,2):>12.4f} {embedding_coverage(ts):>9.3f}')

assert distinct_n(dup, 2) < distinct_n(texts, 2), '重复副本的 distinct-n 更低'
assert self_bleu(dup, 2) > self_bleu(texts, 2), '重复副本的 self-BLEU 更高（越低越多样）'
print('\n✅ 三个度量方向一致：distinct-n 越**高**越多样、self-BLEU 越**低**越多样、覆盖越高越广。')
print('   ⚠️ 但注意「完全随机」的多样性最高、而它作为训练数据毫无价值 ——')
print('   **多样性高 ≠ 有用**。这就是为什么必须有第三个检验。')

## 5 · 检验三：有效性（effectiveness）—— 下游指标真的涨了吗

前两个检验都是**内在的**（不需要训练）。但「增强有没有用」最终只能靠**下游任务**回答。
这里建一个可训练的小分类器，后面每个模块都用它做对照实验。

In [ ]:
VOCAB = sorted(set(NEUTRAL) | POS_WORDS | NEG_WORDS | NEGATORS)
V2I = {w: i for i, w in enumerate(VOCAB)}

def featurize(tokens):
    '''词袋 + 二元特征（能捕捉「否定词 + 正面词」的组合）。'''
    x = np.zeros(len(VOCAB) + 1)
    for t in tokens:
        if t in V2I: x[V2I[t]] += 1.0
    x[-1] = 1.0
    return x

def train_logreg(X, y, epochs=300, lr=0.3, l2=1e-3, seed=0):
    r = np.random.default_rng(seed)
    w = r.normal(size=X.shape[1]) * 0.01
    for _ in range(epochs):
        p = 1 / (1 + np.exp(-(X @ w)))
        g = X.T @ (p - y) / len(y) + l2 * w
        w -= lr * g
    return w

def evaluate(w, X, y):
    return float(((X @ w > 0).astype(int) == y).mean())

def build_xy(pairs_or_data):
    X = np.stack([featurize(t) for t, _ in pairs_or_data])
    y = np.array([lab for _, lab in pairs_or_data])
    return X, y

def effectiveness(train_data, aug_data, test_data, seeds=range(8)):
    '''返回 (baseline 均值, 增强后均值, 每个种子的差值)。'''
    Xte, yte = build_xy(test_data)
    base, aug, diffs = [], [], []
    for s in seeds:
        Xb, yb = build_xy(train_data)
        wb = train_logreg(Xb, yb, seed=s)
        Xa, ya = build_xy(train_data + aug_data)
        wa = train_logreg(Xa, ya, seed=s)
        b, a = evaluate(wb, Xte, yte), evaluate(wa, Xte, yte)
        base.append(b); aug.append(a); diffs.append(a - b)
    return float(np.mean(base)), float(np.mean(aug)), diffs

train = [make_sentence(np.random.default_rng(s)) for s in range(120)]
test  = [make_sentence(np.random.default_rng(10_000 + s)) for s in range(600)]

# 一个安全的增强：只做「中性词的同义替换」（不动否定词与情感词）
SAFE_SYN = {'这家': '本', '店': '餐厅', '菜': '菜品', '服务': '服务员',
            '环境': '氛围', '价格': '收费', '感觉': '觉得', '整体': '总体'}
def safe_synonym(tokens, p, rng):
    return [SAFE_SYN.get(t, t) if (t in SAFE_SYN and rng.random() < p) else t for t in tokens]

r = np.random.default_rng(3)
aug_safe = [(safe_synonym(t, 0.5, r), y) for t, y in train]
b, a, diffs = effectiveness(train, aug_safe, test)
print(f'baseline {b:.4f} -> 增强后 {a:.4f} | 平均差值 {np.mean(diffs):+.4f} ± {np.std(diffs):.4f}')
print(f'8 个种子的差值: {[round(d,4) for d in diffs]}')

# 关键：差值的方差往往与效应同量级 —— 单次实验不可信
assert len(diffs) == 8
print(f'\n⚠️  差值的标准差 {np.std(diffs):.4f} 与效应 {abs(np.mean(diffs)):.4f} 同量级。')
print('   **单个种子的结果毫无意义** —— 模块 05 会给出正确的检验方法。')

## 6 · 三重检验合起来：为什么必须三个都看

In [ ]:
def full_check(name, augment_fn, train_data, test_data, seed=0):
    r = np.random.default_rng(seed)
    pairs = [(t, y, augment_fn(t, r)) for t, y in train_data]
    aug = [(aug_t, y) for _, y, aug_t in pairs]
    fid = fidelity(pairs)
    div = distinct_n([a for a, _ in aug], 2)
    b, a_, diffs = effectiveness(train_data, aug, test_data)
    return {'name': name, '保真度': fid, 'distinct-2': div,
            'Δ准确率': float(np.mean(diffs)), 'Δ标准差': float(np.std(diffs))}

candidates = [
    ('不增强（复制原样本）', lambda t, r: list(t)),
    ('安全同义替换',         lambda t, r: safe_synonym(t, 0.5, r)),
    ('随机删除 p=0.1',       lambda t, r: random_deletion(t, 0.1, r)),
    ('随机删除 p=0.4',       lambda t, r: random_deletion(t, 0.4, r)),
    ('全部打散成随机词',      lambda t, r: list(r.choice(NEUTRAL, size=len(t)))),
]
rows = [full_check(n, f, train, test) for n, f in candidates]
print(f"{'方案':<22s} {'保真度':>8s} {'distinct-2':>11s} {'Δ准确率':>9s} {'Δ标准差':>9s}")
for x in rows:
    print(f'{x["name"]:<22s} {x["保真度"]:>8.1%} {x["distinct-2"]:>11.4f} '
          f'{x["Δ准确率"]:>+9.4f} {x["Δ标准差"]:>9.4f}')

by = {x['name']: x for x in rows}
assert by['不增强（复制原样本）']['保真度'] == 1.0
assert by['随机删除 p=0.4']['保真度'] < by['随机删除 p=0.1']['保真度']
# 「全部打散」多样性最高、保真度最低 —— 完美说明为什么不能只看多样性
assert by['全部打散成随机词']['保真度'] < 0.80, '打散会大量破坏标签'
assert by['不增强（复制原样本）']['distinct-2'] <= by['安全同义替换']['distinct-2'] + 1e-9,     '同义替换的多样性不低于纯复制'
print('\n✅ 三个检验缺一不可：')
print('   · 只看多样性 -> 可能选中「全部打散」（词面变化最大，但标签大量翻转）')
print('   · 只看保真度 -> 会选中「复制原样本」（保真度 100%，但零新信息）')
print('   · 只看下游指标 -> 会被种子噪声骗（Δ标准差常与效应同量级）')
print('\n**保真度 × 多样性 × 有效性，三者同时看，才是判断一个增强的正确方式。**')

## 7 · ✏️ 练习：增强方案的综合打分

实现 `augment_score(fidelity_v, diversity_v, delta_acc, delta_std, min_fidelity=0.95)`：
- 若 `fidelity_v < min_fidelity` → 返回 `0.0`（**保真度是硬门槛，不达标直接否决**）
- 否则若 `delta_acc <= delta_std` → 返回 `0.0`（效应没超过噪声，视为无效）
- 否则返回 `delta_acc * diversity_v`（有效且多样才得分）

In [ ]:
def augment_score(fidelity_v, diversity_v, delta_acc, delta_std, min_fidelity=0.95):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习自测 ——
assert augment_score(0.90, 0.5, 0.02, 0.005) == 0.0, '保真度不达标 -> 直接否决'
assert augment_score(0.99, 0.5, 0.002, 0.010) == 0.0, '效应没超噪声 -> 无效'
s = augment_score(0.99, 0.5, 0.02, 0.005)
assert s > 0 and abs(s - 0.01) < 1e-9
# 同样效应下，多样性更高得分更高
assert augment_score(0.99, 0.8, 0.02, 0.005) > augment_score(0.99, 0.3, 0.02, 0.005)
print(f"{'方案':<22s} {'综合得分':>10s}")
for x in rows:
    sc = augment_score(x['保真度'], x['distinct-2'], x['Δ准确率'], x['Δ标准差'])
    print(f'{x["name"]:<22s} {sc:>10.5f}')
print('✅ 练习通过：保真度是**硬门槛**，效应必须超过噪声，多样性只是加分项')

---
### 📖 参考答案

In [ ]:
def augment_score(fidelity_v, diversity_v, delta_acc, delta_std, min_fidelity=0.95):
    if fidelity_v < min_fidelity:
        return 0.0
    if delta_acc <= delta_std:
        return 0.0
    return delta_acc * diversity_v

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你实现的每一种增强（EDA 四操作、回译、释义、self-instruct、evol-instruct）
都会过三重检验——**保真度**（标签有没有被破坏，规则可精确判定）、
**多样性**（distinct-n / self-BLEU / 嵌入覆盖）、**有效性**（多种子配对检验，
并与「同预算下加真实数据」对照）。

**接下来五个模块**：01 词面增强 → 02 回译与释义 → 03 LLM 指令数据合成 →
04 增强的质量控制 → 05 增强的评测与消融。

下一站：**模块 01 · 词面增强** —— 最便宜、也最容易把标签搞坏的一类。